## Load packages 



In [ ]:
knitr::opts_chunk$set(echo = TRUE)

# for data reading/manipulation 
library(dplyr)
library(tidyr)
library(readr)
library(tibble)
library(janitor)
library(readxl)
# for spatial data and gis
library(sf)
library(ggmap)
library(ggplot2)
library(ggspatial)
library(ggspatial)
library(spdep)
library(leaflet) 
library(RColorBrewer)
library(tmap)


## Explore the data using ggmap and ggplot (Google API)

### Setting up a an API 


Setting up a connetion to Google's API;


1. First: set up an account with Google.


2. Second: to obtain an API go to [https://cloud.google.com/maps-platform/] and follow the onscreen instructions. This documentation shows you how to input the requisite information (e.g. your API key) into R, and it also shows you a few tools that can help you work with the credentialing. In summary the onscreen instructions are as follows. 
 - click 'get started' 
 - click the credentials tab on the left hand side of the screen 
 - click 'create credentials at the top of the screen (visible by the '+' symbol)
 - Select 'API key'
 - Select copy and now you have your personal API key 
 
 
 *You will be asked to set up a billing account but you will not need to pay to follow along with the below steps. Google Cloud has a limited amount of free storage (200 dollars free a month) so remember to always check your storage use and the usage cost under the 'cost breakdown' panel.*
 

3. To authorise the API you have to enable some of Google's API which are required for the project. This includes 
- Google Maps JavaScript API
- Google Maps Geocoding API
- Google Maps Geolocation API
- Google Maps Places API
- Google Maps Static API

If you do not authorise the above API you will recieve and error message as such "Geocoding Service: This API project is not authorised to use the API". To authorise said APIs go to 
a. https://console.cloud.google.com/apis/dashboard?project=missing-314119
b. Click '+ Enable APIS and SERVICES' at the top of the screen
c. Search for each API listed above and click the 'Enable button' 

Once all APIs are enabled, wait 5 minutes for changes to take place, and then move on to the next step to register your Google API to ggmaps. 


4. Third; you need to tell ggmap about your API by using the 'register_google()' function. It would look something like *register_google(key = "hahfuibfiu324898249dbhsgag")* (this is a fake key). This will then set your API key for the current session, Every time you restart R you will have to request a new API key or you can set it permanently by adding 'write = TRUE', the full code being....

*register_google(key = "[your key]", write = TRUE)*

For more information on how to register your decive to the Google API please visit [https://rdrr.io/cran/ggmap/man/register_google.html]. It is incredibly important to not that each API is private and should not be shared with anyone. Users should also be aware that ggmap has no mechanism with which to safeguard the private key once registered with R. 

----------------------------------------------------------------

The basic idea driving ggmap is to take a downloaded map image, plot it as a context layer using ggplot2, and then plot additional content layers of data, statistics, or models on top of the map. In ggmap, downloading a map as an image and formatting the image for plotting is done with the get_map function. More specifically, the get_map is a wrapper function

It is important to note that when using ggmap, users have to first setup an account with Google, enable the relevant API, and then tell R about the user's setup. Do not worry about doing this for this workshop as we will demonstrate the code below. If you would like to run this in your own time then please refer to the steps above which details how to obtain your API and how to enable the services. 


## For one specific area


In [ ]:
# ## Ariel Map of Surrey
qmplot(longitude, latitude, data = crime, colour = crime_type, size = I(3), darken = .3)
# 

# ## Lets just say you were interested in a specific area (in this example we will use Crawley 002B)
# 
# ## Ariel Map of Crawley 002B
 geocode("Crawley")
# 

Crawley <- c(long = -0.152210, lat = 51.15813)
map <- get_map(Crawley, zoom = 13, scale = 1)
ggmap(map)
# 
ggmap(map) +
  geom_point(aes(longitude, latitude), data = crime) 
# 
# 
# ## Colour the Crime Type
# 

ggmap(map) +
   geom_point(aes(longitude, latitude, colour = crime_type), data = crime) 

# 

 ggmap(map) +
   geom_point(aes(longitude, latitude, size = crime_type, colour = crime_type), data = crime) 


## For one specific crime type 



In [ ]:
#subset for just ASB
subset <- crime[which(crime$crime_type == "Anti-social behaviour"),]

#Create the crawley map again 
map <- ggmap(get_googlemap(center = c(long = -0.152210, lat = 51.15813), 
                           zoom = 10, 
                           maptype = "terrain", 
                           color = "color"))
#print the map
print(map)


#overlay the crime data 
map <- ggmap(get_googlemap(center = c(long = -0.631027, lat = 51.215485), 
                           zoom = 9, 
                           maptype = "terrain", 
                           color = "color")) +
  geom_point(data = subset, aes(x = longitude, y = latitude), 
             colour = "red", size = .5)

#print 
print(map)


#lets make a density map using ggplot
map <- ggmap(get_googlemap(center = c(long = -0.152210, lat = 51.15813), 
                           zoom = 9, 
                           maptype = "terrain", 
                           color = "color")) +
  stat_density2d(aes(x = longitude, y = latitude, fill = ..level..), 
               alpha = .2, 
               bins = 10, data = subset, geom = "polygon")


print(map)


### Binning data

Binning, can be thought of as a two-dimensional histogram (shading of the bins take the heights of the bars). You first need to convert the sf data.frame geometry column into a data.frame with separate x, y columns. 

How do you separate the coordinates? 

Luckily a function to do this already exists [https://github.com/r-spatial/sf/issues/231]. The below code is converting a sfc_point to seperate x, y columns 


In [ ]:
sfc_as_cols <- function(x, names = c("x","y")) {
  stopifnot(inherits(x,"sf") && inherits(sf::st_geometry(x),"sfc_POINT"))
  ret <- sf::st_coordinates(x)
  ret <- tibble::as_tibble(ret)
  stopifnot(length(names) == raster::ncol(ret))
  x <- x[ , !names(x) %in% names]
  ret <- setNames(ret,names)
  dplyr::bind_cols(x,ret)
}

sf_seperate <- sfc_as_cols(sf, c("lng", "lat")) 


ggplot(sf_seperate, aes(lng, lat)) +   
  annotation_map_tile() +
  stat_binhex(bins = 30) +                                           
  scale_fill_gradientn(colours = c("white","red"), name = "Frequency")   


#hexagonal = stat_binhex() 
#rectangle = stat_bin2d()
#heat = stat_density2d()  


### Interactive Maps; Leaflet

Leaflet is one of the most popular open-source JavaScript libraries for interactive maps. For more information you can view this link here [https://rstudio.github.io/leaflet/]


In [ ]:
## Subsetting for just ASB 
asb <- subset(crime, crime_type == "Anti-social behaviour")

m <- leaflet(data = asb) %>%
  addProviderTiles("Stamen.Toner") %>% 
  addMarkers(lng=~longitude, lat=~latitude, popup=~as.character(location), label = ~as.character(location))
m


## Moran's I to Spatial regresion 

1. Moran's I (code + p-value from Topic 4)
2. Simple spatial regression: crime_rate ~ urban_rural_flag + res_count
3. Check residuals → still spatially autocorrelated? 
4. Spatial lag model (1-line spdep solution)
5. Compare to regular lm() → which predicts better?


In [ ]:
library(spdep)

# build neighbours from polygons
nb <- poly2nb(surrey_lsoa)
lw <- nb2listw(nb, style = "W")

moran.test(surrey_lsoa_new$count, lw)


p = 0.003962. Therefore, we can say there is strong spatial autocorrelation. High-crime LSOAs cluster together. That's why neighbor-imputation made sense.



In [ ]:
#add res_count and urban_rural_flag to surrey_lsoa_new (i.e. no missing lsoas)
surrey_lsoa_new <- left_join(surrey_lsoa_new, residential_count, by = c("lsoa21cd"="lsoa"))
surrey_lsoa_new <- surrey_lsoa_new %>% 
  mutate(crime_rate = (count/res_count*1000))

surrey_lsoa_new <- left_join(surrey_lsoa_new, urban_rural, by = "lsoa21cd")

## Step 1: Simple linear model (ignores spatial clustering)
ordinary_lm <- lm(count ~ res_count + urban_rural_flag, data = surrey_lsoa_new)
summary(ordinary_lm)


Population strongly predicts crime counts (p = 0.002). Every extra resident = more crime risk.

**The urban/rural surprise**: Not significant (p = 0.45). Urban LSOAs have more crime because they have MORE PEOPLE, not because they're inherently 'urban'.

High-crime areas cluster together → we need spatial regression!"


In [ ]:
# Step 2: Check residuals for spatial autocorrelation
residuals_lm <- residuals(ordinary_lm)
moran.resid <- moran.test(residuals_lm, lw, alternative = "greater")
print(moran.resid)


Residuals still spatially autocorrelated (p = 0.01). Our model systematically under/over-predicts in certain neighbourhoods. The errors themselves form spatial clusters. 

"**Problem confirmed**: Even after population + urban/rural...

Our model's RESIDUALS still show spatial clustering (p = 0.028).

**What does this mean?**
Red clusters = model consistently UNDER-PREDICTS crime
Blue clusters = model consistently OVER-PREDICTS crime

**These are 'neighbourhood effects' our model misses!**

Our ordinary lm() captured population effects but missed the 'neighbourhood contagion' effect!"


In [ ]:
## Step 3: Spatial lag model (accounts for clustering)
# Create lagged dependent variable
surrey_lsoa_new$W_count <- lag.listw(lw, surrey_lsoa_new$count)

# Fit spatial lag model
spatial_lag <- lm(count ~ res_count + urban_rural_flag + W_count, 
                  data = surrey_lsoa_new)
summary(spatial_lag)


Better! Let's compare...



In [ ]:
## Step 4: Compare models
AIC(ordinary_lm, spatial_lag)


Lower AIC = better model. The spatial lag model (350) beats ordinary lm (354) by 4 points. This confirms adding neighbors' crime improves predictions - spatial structure matters!



In [ ]:
## Step 5: Add predictions to the map 
# Add predictions to map
surrey_lsoa_new$predicted <- predict(ordinary_lm)
surrey_lsoa_new$predicted_spatial <- predict(spatial_lag)

# Map residuals (ordinary model misses clusters)
ggplot(surrey_lsoa_new) +
  geom_sf(aes(fill = residuals(ordinary_lm)), colour = NA) +
  scale_fill_gradient2(low = "blue", mid = "white", high = "red", name = "Residuals") +
  labs(title = "Ordinary Model Residuals: Still Clustered!",
       subtitle = "Red = under-predicted, Blue = over-predicted") +
  theme_minimal()

# Map residuals (spatial model = clean)
ggplot(surrey_lsoa_new) +
  geom_sf(aes(fill = residuals(spatial_lag)), colour = NA) +
  scale_fill_gradient2(low = "blue", mid = "white", high = "red", name = "Residuals") +
  labs(title = "Spatial Lag Residuals: Much Cleaner!",
       subtitle = "Spatial structure accounted for") +
  theme_minimal()


This map shows the Spatial Lag model's prediction errors (residuals) across Surrey Heath LSOAs. Red areas represent under-predictions (more crime than expected), blue areas show over-predictions (less crime than expected), and neutral colors indicate accurate predictions. The key victory is the random scatter pattern, no obvious red/blue clusters remain, proving the spatial model successfully captured neighborhood effects that the ordinary linear model missed. Clean residuals + lower AIC confirm the spatial approach works!



### Other imporant functions 

- Jittering: 

Jittering indeed means just adding random noise to a vector of numeric values, by default this is done in jitter-function by drawing samples from the uniform distribution. The range of values in the jittering is chosen according to the data, if amount-parameter is not provided. It helps to grasp where the density of observations is high. 

There are a few packages that offer this method including the 'geom_jitter' function found in the ggplot2 package [https://ggplot2.tidyverse.org/reference/geom_jitter.htm]. Additionally, the 'rjitter' function under the spatsat package. The function rjitter is generic, with methods for point patterns (described here) and for some other types of geometrical objects. Each of the points in the point pattern X is subjected to an independent random displacement. More information can be found here; [https://rdrr.io/cran/spatstat.geom/man/rjitter.html]

-  st_intersect():

This function is also under the sf package and is used to intersect two objects between two sets of objects. More information can be found here [https://r-spatial.github.io/sf/reference/geos_binary_ops.htm]




Data Citation: 

1.	UK Police API. (2026). Street-level crime data. data.police.uk. Retrieved February 27, 2026, from https://data.police.uk
2.	UK Data Service Census Support. (2021). England LSOA boundaries 2021. Retrieved from https://borders.ukdataservice.ac.uk 
3.	Office for National Statistics. (2022). Census 2021 Table TS001: Number of usual residents. Nomis/UK Data Service. Retrieved from https://statistics.ukdataservice.ac.uk
4.	Office for National Statistics. (2022). Census 2021 Table TS066: Economic acitivty. Nomis/UK Data Service. Retrieved from https://statistics.ukdataservice.ac.uk
5.	Office for National Statistics. (2021). Rural Urban Classification (2021) of LSOAs in England and Wales. Open Geography Portal. Retrieved from https://geoportal.statistics.gov.uk 
